# OsteoClassAI v2 · 15 Destilación: un solo modelo que imite al ensamble

**Profesor:** el ensamble de A, B, C y D (mismos pesos que en el cuaderno 09) aplicado a las imágenes de entrenamiento.
**Alumno:** la misma receta que D (ConvNeXt-Tiny, 224 px, SORD + TTA, aumentos de robustez), con la pérdida mitad SORD y mitad probabilidades del profesor (`--teacher`, `--kd-weight 0.5`). Así la comparación con D aísla el efecto de la destilación.

El modelo se elige con la validación de Chen y la prueba se evalúa una vez. Después: comparación pareada con D y Gornale con el detector (exploratorio).

*Entorno de ejecución:* GPU T4. Luego *Ejecutar todas* (~30 min).

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/OsteoClassAI/runs'

In [ ]:
import os
if os.path.isdir('/content/osteoclassai'):
    !cd /content/osteoclassai && git pull -q
else:
    !git clone -q https://github.com/walterandres23/osteoclassai.git /content/osteoclassai
%cd /content/osteoclassai/v2
!pip -q install -e . timm kagglehub imagehash h5py opencv-python-headless
import sys; sys.path.insert(0, '/content/osteoclassai/v2/src')

In [ ]:
import kagglehub, pathlib
path = pathlib.Path(kagglehub.dataset_download('shashwatwork/knee-osteoarthritis-dataset-with-severity'))
root = next(p.parent for p in path.rglob('train') if p.is_dir())
!python -m osteoclass.data --root "$root" --out outputs/data > data.log && tail -n 3 data.log

## Profesor: probabilidades del ensamble en las imágenes de entrenamiento

In [ ]:
!python -m osteoclass.distill --root "$root" --splits outputs/data/splits.csv --runs $RUNS/convnext_tiny_sord $RUNS/convnext_tiny_sord_288 $RUNS/convnext_tiny_sord_288_robust $RUNS/convnext_tiny_sord_robust --out outputs/teacher_train.csv
!mkdir -p /content/drive/MyDrive/OsteoClassAI/distill && cp outputs/teacher_train.csv /content/drive/MyDrive/OsteoClassAI/distill/

## Alumno: receta de D + destilación

In [ ]:
!python -m osteoclass.train --root "$root" --splits outputs/data/splits.csv --model convnext_tiny --loss sord --tta --aug robust --epochs 30 --patience 8 --teacher outputs/teacher_train.csv --kd-weight 0.5 --out outputs/runs/convnext_tiny_sord_robust_kd
!cp -r outputs/runs/convnext_tiny_sord_robust_kd $RUNS/

## Comparación pareada con D en la prueba de Chen (bootstrap por pacientes)

In [ ]:
import json, pandas as pd, numpy as np
from osteoclass.ensemble import paired_bootstrap
def run(name):
    r = json.load(open(f'{RUNS}/{name}/results.json'))
    p = pd.read_csv(f'{RUNS}/{name}/pred_test.csv').sort_values('path').reset_index(drop=True)
    return p, r['expected_grade_cuts'], r['test_cuts']
pD, cD, tD = run('convnext_tiny_sord_robust'); pS, cS, tS = run('convnext_tiny_sord_robust_kd')
assert (pD.path.values == pS.path.values).all()
P = [f'p{k}' for k in range(5)]
d = paired_bootstrap(pD.kl.values, pD[P].values, pS[P].values, pD.patient.astype(str).values, cD, cS, n=2000)
for k in ('qwk', 'accuracy', 'mae_grade', 'oa_auc'):
    print(f"{k:10s} D {tD[k]['value']:.3f}  alumno {tS[k]['value']:.3f}  diferencia {d[k]['value']:+.3f} [{d[k]['ci_low']:+.3f} a {d[k]['ci_high']:+.3f}]")
print('referencia: ensamble de los cuatro, QWK 0.874, exactitud 0.702, MAE 0.316')

## Gornale con el detector (exploratorio): D y alumno

In [ ]:
# Datos: el zip trae dos .rar (uno por experto)
ZIP = '/content/drive/MyDrive/OsteoClassAI/datos_externos/Digital Knee X-ray Images.zip'
!mkdir -p /content/gornale && cd /content/gornale && unzip -q -o "$ZIP"
!which unrar > /dev/null || apt-get -qq install -y unrar > /dev/null
!cd "/content/gornale/Digital Knee X-ray Images/Knee X-ray Images" && for f in *.rar; do unrar x -o+ -idq "$f"; done
root = '/content/gornale/Digital Knee X-ray Images/Knee X-ray Images'
!find "$root" -name '*.png' | wc -l   # debe dar 3300

In [ ]:
!python -m osteoclass.external --root "$root" --runs $RUNS/convnext_tiny_sord_robust $RUNS/convnext_tiny_sord_robust_kd --exploratory --detector $RUNS/detector/best.pt --out outputs/external/gornale_kd
!cp -r outputs/external/gornale_kd /content/drive/MyDrive/OsteoClassAI/external/